# Project 1 - Sargent

In [1]:
# Installs
import pandas as pd
from sqlalchemy import create_engine, text

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [8]:
# Engine - had issues with mysql, using sqlite due to time constraints
conn_string = "sqlite:///store_demo.db"
engine = create_engine(conn_string, echo=False)

In [10]:
# Helper Functions from Lab 01
def run_query (q):
    with engine.connect() as conn:
        df = pd.read_sql(text(q), conn)
        return df

def run_query_commit(q):
    with engine.connect() as conn:
        conn.execute(text("PRAGMA foreign_keys = ON;")) # foreign key for sqlite - found on sqlite site
        conn.execute(text(q))
        conn.commit()

In [11]:
# Create database - simple storage for the demo
DB = "store_demo"

In [13]:
# Create Tables
create_customer = """
CREATE TABLE customer (
  customer_id INTEGER PRIMARY KEY AUTOINCREMENT,
  full_name TEXT NOT NULL,
  email TEXT NOT NULL UNIQUE,
  state TEXT NOT NULL,
  created_at TEXT DEFAULT CURRENT_TIMESTAMP
);
"""

create_product = """
CREATE TABLE product (
  product_id INTEGER PRIMARY KEY AUTOINCREMENT,
  sku TEXT NOT NULL UNIQUE,
  product_name TEXT NOT NULL,
  category TEXT NOT NULL,
  unit_price REAL NOT NULL,
  active INTEGER DEFAULT 1
);
"""

create_orders = """
CREATE TABLE orders (
  order_id INTEGER PRIMARY KEY AUTOINCREMENT,
  customer_id INTEGER NOT NULL,
  order_date TEXT NOT NULL,
  status TEXT NOT NULL,
  FOREIGN KEY (customer_id) REFERENCES customer(customer_id)
);
"""

create_order_item = """
CREATE TABLE order_item (
  order_id INTEGER NOT NULL,
  product_id INTEGER NOT NULL,
  quantity INTEGER NOT NULL,
  unit_price REAL NOT NULL,
  PRIMARY KEY (order_id, product_id),
  FOREIGN KEY (order_id) REFERENCES orders(order_id),
  FOREIGN KEY (product_id) REFERENCES product(product_id)
);
"""

In [14]:
# runquery for tables
run_query_commit(create_customer)
run_query_commit(create_product)
run_query_commit(create_orders)
run_query_commit(create_order_item)

In [15]:
# Create indexes
run_query_commit("CREATE INDEX idx_orders_customer_id ON orders(customer_id);")
run_query_commit("CREATE INDEX idx_orders_order_date ON orders(order_date);")
run_query_commit("CREATE INDEX idx_item_product_id ON order_item(product_id);")

In [18]:
# Load dummy data - generated from ChatGPT
customer_df = pd.read_csv("customer.csv")
product_df  = pd.read_csv("product.csv")
orders_df   = pd.read_csv("orders.csv")
order_item_df = pd.read_csv("order_item.csv")

In [19]:
# Convert for SQLite
customer_df.to_sql("customer", engine, if_exists="append", index=False)
product_df.to_sql("product", engine, if_exists="append", index=False)
orders_df.to_sql("orders", engine, if_exists="append", index=False)
order_item_df.to_sql("order_item", engine, if_exists="append", index=False)

10

In [21]:
# Check to see everything loaded correctly
display(run_query("SELECT COUNT(*) AS customers FROM customer;"))
display(run_query("SELECT COUNT(*) AS products FROM product;"))
display(run_query("SELECT COUNT(*) AS orders FROM orders;"))
display(run_query("SELECT COUNT(*) AS order_items FROM order_item;"))

,customers
0,5


,products
0,5


,orders
0,6


,order_items
0,10


### Queries

In [23]:
# 1 Check the orders with the customer information
q1 = """
SELECT
  o.order_id,
  o.order_date,
  o.status,
  c.full_name,
  c.state
FROM orders o
JOIN customer c USING (customer_id)
ORDER BY o.order_date, o.order_id;
"""
run_query(q1)

,order_id,order_date,status,full_name,state
0,101,2026-01-10,PAID,Ava Chen,CO
1,102,2026-01-20,PAID,Ava Chen,CO
2,103,2026-01-22,REFUNDED,Noah Patel,MA
3,104,2026-02-01,PAID,Mia Rivera,NY
4,105,2026-02-03,PAID,Liam O'Connor,CA
5,106,2026-02-05,PAID,Sophia Kim,WA


In [25]:
# 2 Total value per order - SUM
q2 = """
SELECT
  oi.order_id,
  SUM(oi.quantity * oi.unit_price) AS order_total
FROM order_item oi
GROUP BY oi.order_id
ORDER BY order_total DESC;
"""
run_query(q2)

,order_id,order_total
0,101,61.0
1,102,57.0
2,103,50.0
3,106,50.0
4,104,48.0
5,105,33.0


In [27]:
# 3 Customer metrics
q3 = """
WITH order_totals AS (
    SELECT
        oi.order_id,
        SUM(oi.quantity * oi.unit_price) AS order_total
    FROM order_item oi
    GROUP BY oi.order_id
)
SELECT
    c.customer_id,
    c.full_name,
    COUNT(DISTINCT o.order_id) AS num_orders,
    SUM(ot.order_total) AS total_spend,
    AVG(ot.order_total) AS avg_order_value
FROM customer c
JOIN orders o USING(customer_id)
JOIN order_totals ot USING (order_id)
GROUP BY c.customer_id, c.full_name
ORDER BY total_spend DESC;
"""
run_query(q3)


,customer_id,full_name,num_orders,total_spend,avg_order_value
0,1,Ava Chen,2,118.0,59.0
1,2,Noah Patel,1,50.0,50.0
2,5,Sophia Kim,1,50.0,50.0
3,3,Mia Rivera,1,48.0,48.0
4,4,Liam O'Connor,1,33.0,33.0


In [33]:
# 4  Product Performance
q4 = """
SELECT
    p.product_id,
    p.product_name,
    p.category,
    SUM(oi.quantity) AS units_sold,
    SUM(oi.quantity * oi.unit_price) AS revenue,
    MIN(order_date) AS first_sold_date,
    MAX(order_date) AS last_sold_date
FROM product p
JOIN order_item oi USING (product_id)
JOIN orders USING (order_id)
GROUP BY p.product_id, p.product_name, p.category
ORDER BY revenue DESC;
"""
run_query(q4)

,product_id,product_name,category,units_sold,revenue,first_sold_date,last_sold_date
0,4,Hardcover Sketchbook,Art,7,87.5,2026-01-20,2026-02-05
1,2,Rechargeable Bike Light,Cycling,3,75.0,2026-01-10,2026-01-22
2,1,Insulated Water Bottle,Outdoors,4,72.0,2026-01-10,2026-02-03
3,3,Thermal Cycling Gloves,Cycling,3,45.0,2026-02-01,2026-02-03
4,5,Mechanical Pencil Set,Art,2,19.5,2026-01-20,2026-01-20


In [34]:
# 5 Customers above average spend - nested subquery
q5 = """
SELECT
    c.customer_id,
    c.full_name,
    SUM(oi.quantity * oi.unit_price) AS total_spend
FROM customer c
JOIN orders o USING (customer_id)
JOIN order_item oi USING (order_id)
GROUP BY c.customer_id, c.full_name
HAVING total_spend > (
    SELECT AVG(customer_spend)
    FROM (
    SELECT
        SUM(oi2.quantity *oi2.unit_price) AS customer_spend
    FROM customer c2
    JOIN orders o2 USING (customer_id)
    JOIN order_item oi2 USING (order_id)
    GROUP BY c2.customer_id
  ) t
)
ORDER BY total_spend DESC;
"""
run_query(q5)

,customer_id,full_name,total_spend
0,1,Ava Chen,118.0
